# Training Scaffold

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# -----------------------
# 1. Dataset
# -----------------------
class WheelHeatmapDataset(Dataset):
    def __init__(self, json_path, img_dir, img_size=128, sigma=3):
        with open(json_path) as f:
            self.labels = json.load(f)
        self.img_dir = Path(img_dir)
        self.img_size = img_size
        self.sigma = sigma
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        entry = self.labels[idx]
        img_path = self.img_dir / entry["filename"]
        img = Image.open(img_path).convert("RGB")
        img_t = self.transform(img)

        # Heatmap target
        heatmap = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        if entry["is_present"] == 1.0:
            cx = int(entry["center"][0] * self.img_size)
            cy = int(entry["center"][1] * self.img_size)
            heatmap = self._draw_gaussian(heatmap, (cx, cy), self.sigma)

        return img_t, torch.tensor(heatmap).unsqueeze(0)  # (1,H,W)

    def _draw_gaussian(self, heatmap, center, sigma):
        """Draw a 2D Gaussian on the heatmap."""
        x, y = center
        h, w = heatmap.shape
        xs = np.arange(0, w, 1, np.float32)
        ys = np.arange(0, h, 1, np.float32)[:, np.newaxis]
        g = np.exp(-((xs - x)**2 + (ys - y)**2) / (2 * sigma**2))
        heatmap = np.maximum(heatmap, g)
        return heatmap

# -----------------------
# 2. Model (U-Net style, full 128×128 output)
# -----------------------
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: ResNet18 backbone
        self.encoder = models.resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()
        self.enc_layers = list(self.encoder.children())[:-2]
        self.encoder = nn.Sequential(*self.enc_layers)

        # Decoder: upsample back to 128×128
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),  # 4→8
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 2, stride=2),  # 8→16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 2, stride=2),   # 16→32
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 2, stride=2),    # 32→64
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 2, stride=2),    # 64→128
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=1)             # output heatmap
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x  # (B,1,128,128)

# -----------------------
# 3. Training Loop
# -----------------------
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, device="cuda"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, heatmaps in train_loader:
            imgs, heatmaps = imgs.to(device), heatmaps.to(device)
            preds = model(imgs)
            loss = criterion(preds, heatmaps)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        val_loss = 0
        model.eval()
        with torch.no_grad():
            for imgs, heatmaps in val_loader:
                imgs, heatmaps = imgs.to(device), heatmaps.to(device)
                preds = model(imgs)
                loss = criterion(preds, heatmaps)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

# -----------------------
# 4. Usage
# -----------------------
if __name__ == "__main__":
    base_dir = Path.cwd()
    img_dir = base_dir / "labeled_example_data"
    labels_path = img_dir / "labels.json"

    dataset = WheelHeatmapDataset(labels_path, img_dir, img_size=128, sigma=3)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    model = SimpleUNet()
    train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, device="cuda" if torch.cuda.is_available() else "cpu")

/Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/20 | Train Loss: 0.0016 | Val Loss: 0.0012
Epoch 2/20 | Train Loss: 0.0010 | Val Loss: 0.0010
Epoch 3/20 | Train Loss: 0.0008 | Val Loss: 0.0007
Epoch 4/20 | Train Loss: 0.0007 | Val Loss: 0.0006
Epoch 5/20 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 6/20 | Train Loss: 0.0005 | Val Loss: 0.0005
Epoch 7/20 | Train Loss: 0.0004 | Val Loss: 0.0005
Epoch 8/20 | Train Loss: 0.0004 | Val Loss: 0.0005
Epoch 9/20 | Train Loss: 0.0003 | Val Loss: 0.0004
Epoch 10/20 | Train Loss: 0.0003 | Val Loss: 0.0004
Epoch 11/20 | Train Loss: 0.0003 | Val Loss: 0.0004
Epoch 12/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 13/20 | Train Loss: 0.0003 | Val Loss: 0.0005
Epoch 14/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 15/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 16/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 17/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 18/20 | Train Loss: 0.0001 | Val Loss: 0.0003
Epoch 19/20 | Train Loss: 0.0001 | Val Loss: 0.0004
Epoch 20/20 | Train L

In [2]:
model.eval()

SimpleUNet(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tr

In [3]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

all_results = []

with torch.no_grad():
    for imgs, heatmaps in val_loader:
        imgs = imgs.to(device)
        heatmaps = heatmaps.to(device)

        preds = model(imgs)  # (B,1,128,128)

        preds_np = preds.squeeze(1).cpu().numpy()
        gts_np   = heatmaps.squeeze(1).cpu().numpy()

        for pred_hm, gt_hm in zip(preds_np, gts_np):
            py, px = np.unravel_index(np.argmax(pred_hm), pred_hm.shape)
            gy, gx = np.unravel_index(np.argmax(gt_hm), gt_hm.shape)
            err = np.hypot(px - gx, py - gy)

            all_results.append({
                "gt_x": gx, "gt_y": gy,
                "pred_x": px, "pred_y": py,
                "error_px": err
            })

errors = [r["error_px"] for r in all_results]
print(f"✅ Mean pixel error: {np.mean(errors):.2f}")
print(f"📏 Median pixel error: {np.median(errors):.2f}")
print(f"🎯 % within 5px: {np.mean(np.array(errors) <= 5)*100:.1f}%")

✅ Mean pixel error: 2.15
📏 Median pixel error: 1.41
🎯 % within 5px: 94.5%
